# DadaGP Transformer Position Prior

Trains a small causal transformer on the DadaGP distillate to model **P(string, fret | pitch, full fingering history)** — the neural upgrade of the count-based prior, which conditions on only a single hand-position bucket. The residual errors it targets: the remaining 71/29 thinner-string bias, G–B confusions, and valid-alternate choices that need more than one number of context to resolve.

**Runtime: use a GPU** (Runtime → Change runtime type → T4 GPU). Training is ~15–30 min at the default settings; data prep is a one-time ~10–20 min pass saved to Drive.

**Flow:** distillate → fixed-length training windows (saved to Drive) → train with checkpoints (resume-safe) → intrinsic eval vs the count prior on held-out DadaGP → export weights → paste the integration cell into `AudioToTab_VariantEval_v1` to benchmark `prox_viterbi_transformer` against `prox_viterbi_dadagp` on GuitarSet val.

**Model at a glance:** at each step the input is (current note's pitch, previous note's chosen position, chord-boundary flag); the model attends over up to 64 notes of history and predicts the current note's position out of 150 (string, fret) classes, with physically-invalid positions masked. ~1M parameters.


In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
from pathlib import Path
import json, gzip, math, time, random
from collections import Counter, defaultdict
import numpy as np

from google.colab import drive as _gdrive
_gdrive.mount('/content/drive', force_remount=False)

CAPSTONE_ROOT = Path('/content/drive/MyDrive/Capstone')
DISTILL_DIR   = CAPSTONE_ROOT / 'dadagp_distilled'
TP_DIR        = CAPSTONE_ROOT / 'transformer_prior'
TP_DIR.mkdir(parents=True, exist_ok=True)
WINDOWS_PATH  = TP_DIR / 'training_windows.npz'
CKPT_PATH     = TP_DIR / 'tab_transformer_ckpt.pt'
EXPORT_PATH   = TP_DIR / 'tab_transformer_final.pt'
COUNT_PRIOR_PATH = CAPSTONE_ROOT / 'dadagp_position_prior.json.gz'

OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
MAX_FRET_MODEL = 24
MIN_MIDI, MAX_MIDI = 40, 88
N_PITCH = MAX_MIDI - MIN_MIDI + 1          # 49
N_POS   = 6 * (MAX_FRET_MODEL + 1)         # 150 (pos id = string*25 + fret)
BOS_POS = N_POS                            # 150: "no previous note"
POS_VOCAB = N_POS + 1

CTX      = 64          # notes of history the model sees
STRIDE   = 32
VAL_TRACK_FRAC = 0.02  # held-out DadaGP tracks for intrinsic eval
MAX_WINDOWS    = 250_000   # cap for Colab (~8M notes of training signal)

# training hyperparameters
D_MODEL, N_LAYERS, N_HEADS = 128, 4, 4
BATCH, LR, TRAIN_STEPS = 256, 3e-4, 12_000
VAL_EVERY, CKPT_EVERY  = 500, 1_000

# physically valid positions per pitch id
VALID_POS = np.zeros((N_PITCH, N_POS), dtype=bool)
for s, om in enumerate(OPEN_STRING_MIDI):
    for f in range(MAX_FRET_MODEL + 1):
        m = om + f
        if MIN_MIDI <= m <= MAX_MIDI:
            VALID_POS[m - MIN_MIDI, s * 25 + f] = True

shard_files = sorted(DISTILL_DIR.glob('shard_*.jsonl.gz'))
print(f'{len(shard_files)} distillate shards | windows file exists: {WINDOWS_PATH.exists()}')


Mounted at /content/drive
34 distillate shards | windows file exists: False


In [2]:
# ── One-time data prep: distillate -> fixed-length windows (saved to Drive) ──
# Each track becomes three aligned uint8 arrays:
#   pitch_id  = midi - 40
#   pos_id    = string*25 + fret         (the label; also the "previous position" input, shifted)
#   flag      = 1 if the note starts a new onset group else 0
# Windows of CTX notes, stride STRIDE. Val = last VAL_TRACK_FRAC of tracks.

def track_to_arrays(events):
    pitches, poss, flags = [], [], []
    for item in events:
        if item[0] != 'g':
            continue
        group = sorted(item[1], key=lambda sf: OPEN_STRING_MIDI[sf[0]] + sf[1])
        for j, (s, f) in enumerate(group):
            m = OPEN_STRING_MIDI[s] + f
            if not (MIN_MIDI <= m <= MAX_MIDI) or not (0 <= f <= MAX_FRET_MODEL):
                return None            # out-of-range track: skip whole track
            pitches.append(m - MIN_MIDI)
            poss.append(s * 25 + f)
            flags.append(1 if j == 0 else 0)
    return (np.array(pitches, np.uint8), np.array(poss, np.uint8), np.array(flags, np.uint8))

if WINDOWS_PATH.exists():
    print('Windows already prepared - skipping (delete the file to rebuild).')
else:
    rng = random.Random(0)
    train_w, val_w = [], []
    t0 = time.time()
    for fp in shard_files:
        with gzip.open(fp, 'rt') as f:
            for line in f:
                rec = json.loads(line)
                arrs = track_to_arrays(rec['events'])
                if arrs is None or len(arrs[0]) < CTX + 1:
                    continue
                dest = val_w if rng.random() < VAL_TRACK_FRAC else train_w
                p, q, g = arrs
                for st in range(0, len(p) - CTX, STRIDE):
                    dest.append((p[st:st+CTX], q[st:st+CTX], g[st:st+CTX]))
        print(f'{fp.name}: {len(train_w):,} train / {len(val_w):,} val windows '
              f'({time.time()-t0:.0f}s)')
        if len(train_w) >= MAX_WINDOWS:
            print('Hit MAX_WINDOWS cap - stopping early (raise the cap for more data).')
            break
    rng.shuffle(train_w)
    train_w = train_w[:MAX_WINDOWS]
    def stack(ws):
        return (np.stack([w[0] for w in ws]), np.stack([w[1] for w in ws]),
                np.stack([w[2] for w in ws]))
    tr = stack(train_w); va = stack(val_w[:20_000])
    np.savez_compressed(WINDOWS_PATH,
                        train_pitch=tr[0], train_pos=tr[1], train_flag=tr[2],
                        val_pitch=va[0],   val_pos=va[1],   val_flag=va[2])
    print(f'Saved {tr[0].shape[0]:,} train / {va[0].shape[0]:,} val windows -> {WINDOWS_PATH}')

_z = np.load(WINDOWS_PATH)
TRAIN = (_z['train_pitch'], _z['train_pos'], _z['train_flag'])
VAL   = (_z['val_pitch'],   _z['val_pos'],   _z['val_flag'])
print('train windows:', TRAIN[0].shape, '| val windows:', VAL[0].shape)


shard_1.jsonl.gz: 1,276 train / 0 val windows (0s)
shard_2.jsonl.gz: 1,514 train / 20 val windows (0s)
shard_3.jsonl.gz: 5,004 train / 55 val windows (1s)
shard_4.jsonl.gz: 5,383 train / 70 val windows (1s)
shard_5.jsonl.gz: 5,849 train / 70 val windows (1s)
shard_6.jsonl.gz: 6,714 train / 118 val windows (1s)
shard_7.jsonl.gz: 6,898 train / 118 val windows (2s)
shard_8.jsonl.gz: 7,875 train / 163 val windows (2s)
shard_A.jsonl.gz: 89,410 train / 1,888 val windows (7s)
shard_B.jsonl.gz: 178,399 train / 3,686 val windows (12s)
shard_C.jsonl.gz: 263,971 train / 5,604 val windows (17s)
Hit MAX_WINDOWS cap - stopping early (raise the cap for more data).
Saved 250,000 train / 5,604 val windows -> /content/drive/MyDrive/Capstone/transformer_prior/training_windows.npz
train windows: (250000, 64) | val windows: (5604, 64)


In [3]:
# ── Model ─────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

class TabTransformer(nn.Module):
    """Causal transformer: predicts each note's (string, fret) from its pitch,
    the previous note's position, and up to CTX notes of history."""
    def __init__(self, d=D_MODEL, layers=N_LAYERS, heads=N_HEADS, ctx=CTX):
        super().__init__()
        self.ctx = ctx
        self.emb_pitch  = nn.Embedding(N_PITCH, d)
        self.emb_prev   = nn.Embedding(POS_VOCAB, d)   # previous position (BOS_POS at start)
        self.emb_flag   = nn.Embedding(2, d)
        self.emb_time   = nn.Embedding(ctx, d)
        layer = nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=4*d,
                                           dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.head = nn.Linear(d, N_POS)

    def forward(self, pitch, prev_pos, flag):
        # pitch/prev_pos/flag: (B, T) long tensors
        B, T = pitch.shape
        t_idx = torch.arange(T, device=pitch.device).unsqueeze(0).expand(B, T)
        x = (self.emb_pitch(pitch) + self.emb_prev(prev_pos)
             + self.emb_flag(flag) + self.emb_time(t_idx))
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=pitch.device), 1)
        h = self.encoder(x, mask=causal)
        return self.head(h)                            # (B, T, N_POS)

VALID_POS_T = torch.tensor(VALID_POS, dtype=torch.bool, device=DEVICE)  # (N_PITCH, N_POS)

def masked_logits(logits, pitch):
    return logits.masked_fill(~VALID_POS_T[pitch], -1e9)

def make_batch(arrs, idx, device=None):
    p = torch.tensor(arrs[0][idx], dtype=torch.long)
    q = torch.tensor(arrs[1][idx], dtype=torch.long)
    g = torch.tensor(arrs[2][idx], dtype=torch.long)
    prev = torch.cat([torch.full((q.shape[0], 1), BOS_POS, dtype=torch.long), q[:, :-1]], dim=1)
    dev = device or DEVICE
    return p.to(dev), prev.to(dev), g.to(dev), q.to(dev)

model = TabTransformer().to(DEVICE)
print(sum(t.numel() for t in model.parameters()) / 1e6, 'M parameters')

# 5-second smoke test before committing to training
_p, _prev, _g, _q = make_batch(TRAIN, np.arange(4))
with torch.no_grad():
    out = model(_p, _prev, _g)
assert out.shape == (4, CTX, N_POS)
print('smoke test OK:', out.shape)


device: cuda


/tmp/ipykernel_414/1617340946.py:20: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=layers)


0.846486 M parameters
smoke test OK: torch.Size([4, 64, 150])


In [4]:
# ── Training (checkpointed - safe to disconnect and rerun) ───────────────────
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()
start_step = 0
if CKPT_PATH.exists():
    ck = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    start_step = ck['step']
    print(f'Resumed from step {start_step}')

def val_metrics(n_batches=20):
    model.eval(); tot_loss, tot_acc, tot_n = 0.0, 0, 0
    with torch.no_grad():
        for b in range(n_batches):
            idx = np.random.default_rng(b).integers(0, VAL[0].shape[0], BATCH)
            p, prev, g, q = make_batch(VAL, idx)
            lg = masked_logits(model(p, prev, g), p)
            tot_loss += loss_fn(lg.reshape(-1, N_POS), q.reshape(-1)).item()
            tot_acc  += (lg.argmax(-1) == q).sum().item()
            tot_n    += q.numel()
    model.train()
    return tot_loss / n_batches, tot_acc / tot_n

rng = np.random.default_rng(start_step)
model.train(); t0 = time.time()
for step in range(start_step + 1, TRAIN_STEPS + 1):
    idx = rng.integers(0, TRAIN[0].shape[0], BATCH)
    p, prev, g, q = make_batch(TRAIN, idx)
    logits = masked_logits(model(p, prev, g), p)
    loss = loss_fn(logits.reshape(-1, N_POS), q.reshape(-1))
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % VAL_EVERY == 0:
        vl, va = val_metrics()
        print(f'step {step:6d} | train loss {loss.item():.3f} | '
              f'val loss {vl:.3f} | val top-1 {va:.3f} | {time.time()-t0:.0f}s')
    if step % CKPT_EVERY == 0:
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'step': step}, CKPT_PATH)
torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'step': TRAIN_STEPS}, CKPT_PATH)
print('training complete')


step    500 | train loss 0.143 | val loss 0.120 | val top-1 0.956 | 27s
step   1000 | train loss 0.106 | val loss 0.095 | val top-1 0.965 | 54s
step   1500 | train loss 0.093 | val loss 0.087 | val top-1 0.968 | 82s
step   2000 | train loss 0.093 | val loss 0.082 | val top-1 0.969 | 110s
step   2500 | train loss 0.080 | val loss 0.079 | val top-1 0.970 | 140s
step   3000 | train loss 0.076 | val loss 0.078 | val top-1 0.971 | 170s
step   3500 | train loss 0.083 | val loss 0.075 | val top-1 0.972 | 200s
step   4000 | train loss 0.068 | val loss 0.075 | val top-1 0.972 | 230s
step   4500 | train loss 0.076 | val loss 0.074 | val top-1 0.973 | 261s
step   5000 | train loss 0.066 | val loss 0.072 | val top-1 0.973 | 291s
step   5500 | train loss 0.088 | val loss 0.071 | val top-1 0.973 | 321s
step   6000 | train loss 0.056 | val loss 0.071 | val top-1 0.974 | 351s
step   6500 | train loss 0.070 | val loss 0.070 | val top-1 0.974 | 382s
step   7000 | train loss 0.065 | val loss 0.069 | val 

In [5]:
# ── Intrinsic eval: transformer vs count prior on held-out DadaGP ────────────
# Same question for both models: given the pitch and the history, predict the
# position. Count prior sees only (pitch, hand bucket); transformer sees 64 notes.

class CountPrior:
    TAU, KAPPA = 20.0, 5.0
    def __init__(self, path):
        with gzip.open(path, 'rt') as f:
            raw = json.load(f)
        parse = lambda k: tuple(int(x) for x in k.split(','))
        self.uncond = {int(m): {parse(k): v for k, v in c.items()} for m, c in raw['uncond'].items()}
        self.cond   = {parse(k): {parse(kk): v for kk, v in c.items()} for k, c in raw['cond'].items()}
        self._ut = {m: sum(c.values()) for m, c in self.uncond.items()}
        self._ct = {k: sum(c.values()) for k, c in self.cond.items()}
    def argmax(self, midi, bucket):
        playable = [(s, midi - om) for s, om in enumerate(OPEN_STRING_MIDI)
                    if 0 <= midi - om <= MAX_FRET_MODEL]
        u = 1.0 / len(playable)
        uc = self.uncond.get(midi, {})
        def p(pos):
            base = (uc.get(pos, 0) + self.KAPPA * u) / (self._ut.get(midi, 0) + self.KAPPA)
            cc = self.cond.get((midi, bucket))
            if not cc: return base
            return (cc.get(pos, 0) + self.TAU * base) / (self._ct[(midi, bucket)] + self.TAU)
        return max(playable, key=p)

count_prior = CountPrior(COUNT_PRIOR_PATH)

n_eval_windows = min(2000, VAL[0].shape[0])
idx = np.arange(n_eval_windows)
p, prev, g, q = make_batch(VAL, idx)
model.eval()
with torch.no_grad():
    pred_t = masked_logits(model(p, prev, g), p).argmax(-1).cpu().numpy()
q_np, p_np, g_np = VAL[1][idx], VAL[0][idx], VAL[2][idx]

# count-prior predictions with its bucket recomputed from the running history
tf_correct = cp_correct = n_scored = 0
for w in range(n_eval_windows):
    bucket = -1; cur_frets = []
    for t in range(CTX):
        midi = int(p_np[w, t]) + MIN_MIDI
        if g_np[w, t] == 1 and cur_frets:                 # new onset group starts
            fr = [f for f in cur_frets if f > 0]
            if fr: bucket = int(min(max(round(np.mean(fr)), 0), 12))
            cur_frets = []
        if t >= CTX // 2:                                  # score second half only (warm context)
            s_c, f_c = count_prior.argmax(midi, bucket)
            cp_correct += int(s_c * 25 + f_c == int(q_np[w, t]))
            tf_correct += int(int(pred_t[w, t]) == int(q_np[w, t]))
            n_scored += 1
        cur_frets.append(int(q_np[w, t]) % 25)
print(f'held-out DadaGP next-position top-1 accuracy ({n_scored:,} notes):')
print(f'  count prior (bucket context): {cp_correct / n_scored:.3f}')
print(f'  transformer (64-note context): {tf_correct / n_scored:.3f}')


held-out DadaGP next-position top-1 accuracy (64,000 notes):
  count prior (bucket context): 0.857
  transformer (64-note context): 0.976


In [6]:
# ── Export weights + config for the eval notebook ─────────────────────────────
torch.save({'model': model.state_dict(),
            'config': {'d': D_MODEL, 'layers': N_LAYERS, 'heads': N_HEADS, 'ctx': CTX}},
           EXPORT_PATH)
print('exported:', EXPORT_PATH)


exported: /content/drive/MyDrive/Capstone/transformer_prior/tab_transformer_final.pt


## Integration cell — PASTE INTO `AudioToTab_VariantEval_v1`

Paste the next cell after Section 10d (it needs `candidate_groups_voiced`, `group_notes_by_onset`, `CAGED_WEIGHTS`, `make_assignment_eval`), then add to the methods dict:

```python
'prox_viterbi_transformer': make_assignment_eval(assign_prox_viterbi_transformer, 'prox_viterbi_transformer'),
```

Because the transformer conditions on the full decoded history, exact Viterbi no longer applies — this decoder uses **beam search** (width 8) with the transformer scoring each candidate fingering given the beam's history. **Run the eval notebook on a GPU runtime** for this method; on CPU it works but is slow (~1–2 min/recording).

Head-to-head readout vs `prox_viterbi_dadagp`: overall given-pitch accuracy, the solo segment, and the thinner-string direction share — the transformer's richer context should compress the residual 71/29 bias if it's earning its keep.


In [7]:
# ============================================================
# prox-Viterbi -> BEAM SEARCH with transformer position scoring
# PASTE INTO AudioToTab_VariantEval_v1 (after Section 10d).
# ============================================================
import torch as _torch
import torch.nn as _nn

TRANSFORMER_PRIOR_PATH = CAPSTONE_ROOT / 'transformer_prior' / 'tab_transformer_final.pt'
TRANSFORMER_WEIGHT = 1.75      # same role as DADAGP_PRIOR_WEIGHT; sweep on val
BEAM_WIDTH = 8

_TP_MIN_MIDI, _TP_N_PITCH, _TP_N_POS = 40, 49, 150
_TP_BOS = 150
_TP_DEVICE = 'cuda' if _torch.cuda.is_available() else 'cpu'

class _TabTransformer(_nn.Module):
    def __init__(self, d, layers, heads, ctx):
        super().__init__()
        self.ctx = ctx
        self.emb_pitch = _nn.Embedding(_TP_N_PITCH, d)
        self.emb_prev  = _nn.Embedding(_TP_N_POS + 1, d)
        self.emb_flag  = _nn.Embedding(2, d)
        self.emb_time  = _nn.Embedding(ctx, d)
        layer = _nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=4*d,
                                            dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = _nn.TransformerEncoder(layer, num_layers=layers)
        self.head = _nn.Linear(d, _TP_N_POS)
    def forward(self, pitch, prev_pos, flag):
        B, T = pitch.shape
        t_idx = _torch.arange(T, device=pitch.device).unsqueeze(0).expand(B, T)
        x = (self.emb_pitch(pitch) + self.emb_prev(prev_pos)
             + self.emb_flag(flag) + self.emb_time(t_idx))
        causal = _torch.triu(_torch.ones(T, T, dtype=_torch.bool, device=pitch.device), 1)
        return self.head(self.encoder(x, mask=causal))

_ck = _torch.load(TRANSFORMER_PRIOR_PATH, map_location=_TP_DEVICE)
_cfg = _ck['config']
TP_MODEL = _TabTransformer(_cfg['d'], _cfg['layers'], _cfg['heads'], _cfg['ctx']).to(_TP_DEVICE)
TP_MODEL.load_state_dict(_ck['model']); TP_MODEL.eval()
_TP_CTX = _cfg['ctx']

_VALID = np.zeros((_TP_N_PITCH, _TP_N_POS), dtype=bool)
for _s, _om in enumerate(OPEN_STRING_MIDI):
    for _f in range(25):
        _m = _om + _f
        if _TP_MIN_MIDI <= _m <= _TP_MIN_MIDI + _TP_N_PITCH - 1:
            _VALID[_m - _TP_MIN_MIDI, _s * 25 + _f] = True
_VALID_T = _torch.tensor(_VALID, dtype=_torch.bool, device=_TP_DEVICE)

def _score_extensions(histories, extensions):
    """histories: list of (pitch_list, pos_list, flag_list) per beam.
    extensions: list of lists - extensions[b] = candidate extensions for beam b,
      each a (pitches, positions, flags) tuple for the new group's notes.
    Returns nll[b][c] = total transformer NLL of extension c under beam b."""
    seq_p, seq_prev, seq_g, meta = [], [], [], []
    for b, (hp, hq, hg) in enumerate(histories):
        for c, (ep, eq, eg) in enumerate(extensions[b]):
            p = (hp + ep)[-_TP_CTX:]
            q = (hq + eq)[-_TP_CTX:]
            g = (hg + eg)[-_TP_CTX:]
            prev = [_TP_BOS] + q[:-1]
            seq_p.append(p); seq_prev.append(prev); seq_g.append(g)
            meta.append((b, c, len(ep), len(p)))
    maxlen = max(len(s) for s in seq_p)
    def pad(seqs, val):
        return _torch.tensor([s + [val] * (maxlen - len(s)) for s in seqs],
                             dtype=_torch.long, device=_TP_DEVICE)
    P, PR, G = pad(seq_p, 0), pad(seq_prev, _TP_BOS), pad(seq_g, 0)
    with _torch.no_grad():
        logits = TP_MODEL(P, PR, G)
        logits = logits.masked_fill(~_VALID_T[P], -1e9)
        logp = _torch.log_softmax(logits, dim=-1)
    out = defaultdict(dict)
    for row, (b, c, n_new, L) in enumerate(meta):
        nll = 0.0
        full_q = (histories[b][1] + [pos for pos in extensions[b][c][1]])[-_TP_CTX:]
        for t in range(L - n_new, L):
            nll -= float(logp[row, t, full_q[t]])
        out[b][c] = nll
    return out

def assign_prox_viterbi_transformer(notes, key=None):
    groups = group_notes_by_onset(notes)
    if not groups:
        return []
    beams = [{'hp': [], 'hq': [], 'hg': [], 'cost': 0.0, 'centers': None, 'choice': []}]
    allc = []
    for g in groups:
        cands = candidate_groups_voiced(g)
        allc.append(cands if cands else None)

    for gi, g in enumerate(groups):
        cands = allc[gi]
        if cands is None:
            for b in beams:
                b['choice'].append(None)
            continue
        g_sorted_idx = sorted(range(len(g)), key=lambda k: g[k]['midi'])
        exts_per_cand = []
        for c in cands:
            pos_sorted = sorted(c['positions'], key=lambda p_: p_['midi'])
            ep = [int(p_['midi']) - _TP_MIN_MIDI for p_ in pos_sorted]
            eq = [int(p_['string']) * 25 + int(p_['fret']) for p_ in pos_sorted]
            eg = [1] + [0] * (len(pos_sorted) - 1)
            exts_per_cand.append((ep, eq, eg))
        histories = [(b['hp'], b['hq'], b['hg']) for b in beams]
        extensions = [exts_per_cand for _ in beams]
        nll = _score_extensions(histories, extensions)

        scored = []
        for bi, b in enumerate(beams):
            for ci, c in enumerate(cands):
                cf = [p_['fret'] for p_ in c['positions'] if p_['fret'] > 0]
                cc = float(np.mean(cf)) if cf else 0.0
                move = 0.0
                if b['centers'] is not None:
                    move = CAGED_WEIGHTS['hand_move'] * abs(cc - b['centers'])
                total = (b['cost'] + c['base_cost'] + move
                         + TRANSFORMER_WEIGHT * nll[bi][ci])
                scored.append((total, bi, ci, cc if cf else b['centers']))
        scored.sort(key=lambda x: x[0])
        new_beams = []
        for total, bi, ci, center in scored[:BEAM_WIDTH]:
            b = beams[bi]; ep, eq, eg = exts_per_cand[ci]
            new_beams.append({
                'hp': (b['hp'] + ep)[-_TP_CTX:], 'hq': (b['hq'] + eq)[-_TP_CTX:],
                'hg': (b['hg'] + eg)[-_TP_CTX:], 'cost': total,
                'centers': center, 'choice': b['choice'] + [ci]})
        beams = new_beams

    best = min(beams, key=lambda b: b['cost'])
    out = []
    for gi, (g, ci) in enumerate(zip(groups, best['choice'])):
        if ci is None or allc[gi] is None:
            continue
        c = allc[gi][ci]
        pos_sorted = sorted(c['positions'], key=lambda p_: p_['midi'])
        g_sorted = sorted(g, key=lambda n_: n_['midi'])
        for note, p_ in zip(g_sorted, pos_sorted):
            row = dict(note)
            row.update({'pred_string': p_['string'], 'pred_fret': p_['fret'],
                        'method': 'prox_viterbi_transformer'})
            out.append(row)
    return sorted(out, key=lambda x: (x['start'], x['midi']))

print(f'prox_viterbi_transformer ready (device={_TP_DEVICE}, beam={BEAM_WIDTH}, W={TRANSFORMER_WEIGHT})')


prox_viterbi_transformer ready (device=cuda, beam=8, W=1.75)


/tmp/ipykernel_414/3020753036.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = _nn.TransformerEncoder(layer, num_layers=layers)
